# EMA Crossover — Signal Research

**Instrument:** QQQ (5-min bars, regular session 9:30–16:00 ET)
**Signal:** Triple EMA alignment (13 / 48 / 200) — EMA crossover only (Strategy 2)
**Stop:** ATR-based (0.05× ATR-14)
**Exit:** Stop loss or EOD (15:55 ET)

This notebook generates raw signal trades — **no position sizing, no equity tracking.**

## 1. Setup

In [1]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit

from _shared.loaders_data import fetch_historical_data
from _shared.fees import calculate_fees_pct
from _shared.significance import full_significance_report, print_significance_report

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Configuration

In [2]:
SYMBOL     = "QQQ"
INSTRUMENT = "QQQ"
START_DATE = "2016-01-01"
END_DATE   = "2026-04-01"

STARTING_CAPITAL = 100_000
STRATEGY_NAME    = "EMA Crossover"
SAVE_NAME        = "ema_crossover"

# Signal parameters (Strategy 2 — EMA crossover only, ATR stop 0.05x)
STOP_TYPE  = "atr"
STOP_VALUE = 0.05

## 3. Data Fetching

In [3]:
data_5min = fetch_historical_data(
    [SYMBOL], TimeFrame(5, TimeFrameUnit.Minute), START_DATE, END_DATE)
data_minute = data_5min[SYMBOL]

data_1d = fetch_historical_data(
    [SYMBOL], TimeFrame(1, TimeFrameUnit.Day), START_DATE, END_DATE)
data_daily = data_1d[SYMBOL]

print(f"5-min bars: {len(data_minute):,}")
print(f"Daily bars: {len(data_daily):,}")

Fetching QQQ...
  456,689 bars
Fetching QQQ...
  2,575 bars
5-min bars: 456,689
Daily bars: 2,575


## 4. Feature Engineering

In [4]:
# EMAs on 5-min bars
data_minute["ema13"]  = data_minute["close"].ewm(span=13,  adjust=False).mean()
data_minute["ema48"]  = data_minute["close"].ewm(span=48,  adjust=False).mean()
data_minute["ema200"] = data_minute["close"].ewm(span=200, adjust=False).mean()

# ATR-14 from daily bars (shifted to avoid look-ahead)
prev_close = data_daily["close"].shift(1)
tr = pd.concat([
    data_daily["high"] - data_daily["low"],
    (data_daily["high"] - prev_close).abs(),
    (data_daily["low"]  - prev_close).abs(),
], axis=1).max(axis=1)
atr14_prev = tr.ewm(span=14, adjust=False).mean().shift(1)
atr_df = atr14_prev.to_frame(name="atr14")
atr_df["date"] = pd.to_datetime(atr_df.index.date)

# Merge ATR into 5-min data
data = data_minute.copy()
data["date"] = pd.to_datetime(data.index.date)
original_index = data.index
data = data.merge(atr_df[["date", "atr14"]], on="date", how="left")
data.index = original_index

print(f"Merged data shape: {data.shape}")

Merged data shape: (456689, 12)


## 5. Signal Generation

In [5]:
def generate_signals(data, stop_type="atr", stop_value=0.05):
    """
    Generate raw EMA crossover trades — signal only, no sizing.

    Signal: EMA13 > EMA48 > EMA200 → long, reverse → short.
    Stop: ATR-based or percentage-based.
    Exit: Stop hit or EOD (15:55 ET).

    Returns DataFrame with standardized columns.
    """
    data = data.dropna(subset=["atr14", "ema13", "ema48", "ema200"])

    market_open  = pd.to_datetime("09:30:00").time()
    market_close = pd.to_datetime("16:00:00").time()
    data_reg = data[(data.index.time >= market_open) & (data.index.time < market_close)].copy()

    # Signals: EMA crossover only (Strategy 2)
    long_sig  = (data_reg["ema13"] > data_reg["ema48"]) & (data_reg["ema48"] > data_reg["ema200"])
    short_sig = (data_reg["ema13"] < data_reg["ema48"]) & (data_reg["ema48"] < data_reg["ema200"])
    data_reg["long_entries"]  = long_sig.shift(1).fillna(False)
    data_reg["short_entries"] = short_sig.shift(1).fillna(False)

    trades = []
    holding = False

    for timestamp, row in data_reg.iterrows():
        if holding:
            # Check stop
            if direction == "long" and row["low"] <= stop_price:
                exit_price, exit_time, exit_reason = stop_price, timestamp, "stop"
                holding = False
            elif direction == "short" and row["high"] >= stop_price:
                exit_price, exit_time, exit_reason = stop_price, timestamp, "stop"
                holding = False
            elif timestamp.hour == 15 and timestamp.minute >= 55:
                exit_price, exit_time, exit_reason = row["close"], timestamp, "eod"
                holding = False
            else:
                continue

            if direction == "long":
                pct_ret = (exit_price - entry_price) / entry_price
            else:
                pct_ret = (entry_price - exit_price) / entry_price

            trades.append({
                "entry_time":      entry_time,
                "exit_time":       exit_time,
                "direction":       direction,
                "instrument":      INSTRUMENT,
                "entry_price":     round(entry_price, 4),
                "exit_price":      round(exit_price, 4),
                "pct_return_gross": round(pct_ret, 6),
                "exit_reason":     exit_reason,
                "stop_price":      round(stop_price, 4),
            })
            continue

        # Entry
        if row.get("long_entries", False) or row.get("short_entries", False):
            direction = "long" if row.get("long_entries", False) else "short"
            entry_price = row["open"]
            entry_time = timestamp

            if stop_type == "atr":
                risk = row["atr14"] * stop_value
            else:
                risk = entry_price * stop_value

            if risk <= 0:
                continue

            if direction == "long":
                stop_price = entry_price - risk
            else:
                stop_price = entry_price + risk

            holding = True

    return pd.DataFrame(trades)

print("generate_signals() defined.")

generate_signals() defined.


## 6. Signal Generation & Significance

In [ ]:
raw_trades = generate_signals(data, stop_type=STOP_TYPE, stop_value=STOP_VALUE)
print(f"Total signal trades: {len(raw_trades)}")

# Net returns
raw_trades["fee_pct"] = raw_trades.apply(
    lambda t: calculate_fees_pct(t["entry_price"], t["exit_price"], t["direction"]), axis=1)
raw_trades["pct_return_net"] = raw_trades["pct_return_gross"] - raw_trades["fee_pct"]

print(f"Avg fee: {raw_trades['fee_pct'].mean()*100:.4f}% per trade")
print(f"Avg gross return: {raw_trades['pct_return_gross'].mean()*100:.4f}%")
print(f"Avg net return: {raw_trades['pct_return_net'].mean()*100:.4f}%")

# Significance: GROSS
sig_gross = raw_trades[["direction", "pct_return_gross"]].copy()
sig_gross["net_pnl"] = sig_gross["pct_return_gross"]
sig_gross["equity_before"] = 1.0
sig_gross["position"] = sig_gross["direction"]

if len(sig_gross) >= 5:
    report_gross = full_significance_report(sig_gross, strategy_name=f"{STRATEGY_NAME} (gross)")
    print_significance_report(report_gross)

# Significance: NET
sig_net = raw_trades[["direction", "pct_return_net"]].copy()
sig_net["net_pnl"] = sig_net["pct_return_net"]
sig_net["equity_before"] = 1.0
sig_net["position"] = sig_net["direction"]

if len(sig_net) >= 5:
    report_net = full_significance_report(sig_net, strategy_name=f"{STRATEGY_NAME} (net)")
    print_significance_report(report_net)

## 7. Simple Equity Curve — Gross vs Net

In [ ]:
BET_SIZE = 0.85

equity_gross = STARTING_CAPITAL
equity_net = STARTING_CAPITAL
gross_curve = [STARTING_CAPITAL]
net_curve = [STARTING_CAPITAL]

for _, trade in raw_trades.iterrows():
    shares_g = int(equity_gross * BET_SIZE / trade["entry_price"])
    shares_n = int(equity_net * BET_SIZE / trade["entry_price"])

    if trade["direction"] == "long":
        pnl_g = shares_g * (trade["exit_price"] - trade["entry_price"])
        pnl_n = shares_n * (trade["exit_price"] - trade["entry_price"])
    else:
        pnl_g = shares_g * (trade["entry_price"] - trade["exit_price"])
        pnl_n = shares_n * (trade["entry_price"] - trade["exit_price"])

    from _shared.fees import calculate_fees
    fees = calculate_fees(shares_n, trade["entry_price"], trade["exit_price"], trade["direction"])

    equity_gross += pnl_g
    equity_net += pnl_n - fees
    gross_curve.append(equity_gross)
    net_curve.append(equity_net)

dates = [pd.Timestamp(START_DATE)] + pd.to_datetime(raw_trades["exit_time"]).dt.tz_localize(None).tolist()

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(dates, gross_curve, linewidth=2, label=f"Gross — {(gross_curve[-1]/STARTING_CAPITAL-1)*100:.1f}%")
ax.plot(dates, net_curve, linewidth=2, label=f"Net (fees) — {(net_curve[-1]/STARTING_CAPITAL-1)*100:.1f}%")
ax.axhline(STARTING_CAPITAL, color="gray", linestyle="--", alpha=0.3)
ax.set_title(f"{STRATEGY_NAME} — Simple Implementation ({BET_SIZE:.0%} bet, 1× leverage)", fontsize=14)
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylabel("Equity ($)")
plt.tight_layout(); plt.show()

print(f"Fee drag: ${gross_curve[-1] - net_curve[-1]:,.0f}")

## 8. Save Standardized Trades

In [ ]:
import os, json

os.makedirs("results", exist_ok=True)

std_cols = ["entry_time", "exit_time", "direction", "instrument",
            "entry_price", "exit_price", "pct_return_gross", "exit_reason", "stop_price"]
raw_trades[std_cols].to_csv("results/ema_crossover_trades.csv", index=False)
print(f"Saved {len(raw_trades)} standardized trades → results/ema_crossover_trades.csv")

summary = {
    "strategy":    STRATEGY_NAME,
    "instrument":  INSTRUMENT,
    "portfolio":   "short_term",
    "period":      f"{START_DATE} → {END_DATE}",
    "params":      {"stop_type": STOP_TYPE, "stop_value": STOP_VALUE, "strategy_variant": 2},
    "trades":      len(raw_trades),
    "significance": {
        "gross": {"sharpe": report_gross["bootstrap"]["observed_sharpe"],
                  "verdict": report_gross["verdict"],
                  "tests_passed": report_gross["tests_passed"]},
        "net":   {"sharpe": report_net["bootstrap"]["observed_sharpe"],
                  "verdict": report_net["verdict"],
                  "tests_passed": report_net["tests_passed"]},
    },
}
with open("results/ema_crossover_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"Saved summary → results/ema_crossover_summary.json")
print(f"\nNext: run EMA_Implementation.ipynb for sizing comparison")